<a href="https://colab.research.google.com/github/EliasNoorzad/temporal-reasoning-TISER/blob/main/notebooks/run_test_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/EliasNoorzad/temporal-reasoning-TISER.git

Cloning into 'temporal-reasoning-TISER'...
remote: Enumerating objects: 208, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 208 (delta 0), reused 0 (delta 0), pack-reused 207 (from 1)
Receiving objects: 100% (208/208), 378.56 KiB | 2.54 MiB/s, done.
Resolving deltas: 100% (129/129), done.


In [2]:
%cd temporal-reasoning-TISER
!ls

/content/temporal-reasoning-TISER
configs  notebooks  README.md  requirements.txt  results  src


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 69.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [5]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [6]:
import gc

from argparse import Namespace
from pathlib import Path

import torch

from src.dataset import load_filtered_test_dataset

from src.evaluate import (
    load_evaluation_model,
    run_combined_prompt_evaluation,
    read_jsonl,
    write_combined_summary,
)

LORA_ADAPTER_PATH = (
    "/content/drive/MyDrive/TISER/checkpoints/train_lora/checkpoint-49040"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/TISER/results/test"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

test_dataset = load_filtered_test_dataset()

print(f"Filtered test examples: {len(test_dataset)}")
print(f"Results will be saved to: {OUTPUT_DIR}")

README.md:   0%|          | 0.00/4.00k [00:00<?, ?B/s]

data/TISER_train.json: reconstructing file:   0%|          |  0.00B /  208MB            

data/TISER_train.json: downloading bytes:           |  0.00B            

data/TISER_test.json: reconstructing file:   0%|          |  0.00B / 88.0MB            

data/TISER_test.json: downloading bytes:           |  0.00B            

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Original test examples: 22014
Test examples <= 2048 tokens: 20442
Removed examples: 1572
Filtered test examples: 20442
Results will be saved to: /content/drive/MyDrive/TISER/results/test


In [7]:
def run_full_test_both(
    model_type="lora",
    lora_adapter_path=None,
    batch_size=16,
):
    args = Namespace(
        model_type=model_type,
        prompt_type="both",
        lora_adapter_path=lora_adapter_path,
        output_dir=str(OUTPUT_DIR),
        direct_max_new_tokens=128,
        tiser_max_new_tokens=2048,
        batch_size=batch_size,
        device_map="auto",
    )

    tokenizer, model = load_evaluation_model(args)

    results_path = OUTPUT_DIR / f"{model_type}_both_results.jsonl"
    summary_path = OUTPUT_DIR / f"{model_type}_both_summary.json"

    run_combined_prompt_evaluation(
        args=args,
        model=model,
        tokenizer=tokenizer,
        test_dataset=test_dataset,
        results_path=results_path,
    )

    records = read_jsonl(results_path)
    summary = write_combined_summary(summary_path, records)

    print("\n==============================")
    print("LORA — DIRECT + TISER FULL TEST")
    print("==============================")
    print(f"Examples: {len(records)}")

    print(f"Direct EM: {summary['direct_overall_em']:.2f}%")
    print(f"Direct F1: {summary['direct_overall_token_f1']:.2f}%")
    print(
        f"Direct avg generated tokens: "
        f"{summary['direct_average_generated_tokens']:.2f}"
    )

    print(f"TISER EM: {summary['tiser_overall_em']:.2f}%")
    print(f"TISER F1: {summary['tiser_overall_token_f1']:.2f}%")
    print(
        f"TISER avg generated tokens: "
        f"{summary['tiser_average_generated_tokens']:.2f}"
    )

    print(f"Saved: {results_path}")
    print(f"Saved: {summary_path}")

    del model
    del tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary

In [8]:
lora_both_summary = run_full_test_both(
    model_type="lora",
    lora_adapter_path=LORA_ADAPTER_PATH,
    batch_size=16,
)

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Generating direct and TISER:   0%|          | 0/20442 [00:00<?, ?it/s]


LORA — DIRECT + TISER FULL TEST
Examples: 20442
Direct EM: 74.83%
Direct F1: 81.27%
Direct avg generated tokens: 5.96
TISER EM: 80.42%
TISER F1: 85.91%
TISER avg generated tokens: 283.00
Saved: /content/drive/MyDrive/TISER/results/test/lora_both_results.jsonl
Saved: /content/drive/MyDrive/TISER/results/test/lora_both_summary.json


In [6]:
import json
import re
from pathlib import Path

from src.evaluate import exact_match, token_f1


INPUT_PATH = Path(
    "/content/drive/MyDrive/TISER/results/test/"
    "lora_both_results.jsonl"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/TISER/results/test"
)

NORMALIZED_RESULTS_PATH = OUTPUT_DIR / "lora_both_normalized_results.jsonl"
NORMALIZED_SUMMARY_PATH = OUTPUT_DIR / "lora_both_normalized_summary.json"


def normalize_tiser_answer(prediction, gold_answer):
    prediction = str(prediction).strip()
    gold_answer = str(gold_answer).strip()

    # Duration questions:
    # Gold answers can contain "year" or "years", while the trained model
    # commonly produces only the numeric duration.
    if re.search(r"\s+years?$", gold_answer, flags=re.IGNORECASE):
        prediction = re.sub(
            r"\s+years?$",
            "",
            prediction,
            flags=re.IGNORECASE,
        ).strip()

        gold_answer = re.sub(
            r"\s+years?$",
            "",
            gold_answer,
            flags=re.IGNORECASE,
        ).strip()

    # Event-boundary questions:
    # Gold answers may use "(event) starts" / "(event) ends",
    # while the model commonly outputs only the event.
    if re.search(r"\s+(starts|ends)$", gold_answer, flags=re.IGNORECASE):
        prediction = re.sub(
            r"\s+(starts|ends)$",
            "",
            prediction,
            flags=re.IGNORECASE,
        ).strip()

        gold_answer = re.sub(
            r"\s+(starts|ends)$",
            "",
            gold_answer,
            flags=re.IGNORECASE,
        ).strip()

        if prediction.startswith("(") and prediction.endswith(")"):
            prediction = prediction[1:-1].strip()

        if gold_answer.startswith("(") and gold_answer.endswith(")"):
            gold_answer = gold_answer[1:-1].strip()

    return prediction, gold_answer


# Load results
records = []

with INPUT_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

print(f"Loaded examples: {len(records)}")


normalized_records = []

changed_predictions = 0
raw_errors_fixed = 0

for record in records:
    new_record = dict(record)

    raw_prediction = record["tiser_answer"]
    raw_gold = record["gold_answer"]

    normalized_prediction, normalized_gold = normalize_tiser_answer(
        raw_prediction,
        raw_gold,
    )

    if normalized_prediction != raw_prediction:
        changed_predictions += 1

    normalized_em = exact_match(
        normalized_prediction,
        normalized_gold,
    )

    normalized_f1 = token_f1(
        normalized_prediction,
        normalized_gold,
    )

    if not record["tiser_em"] and normalized_em:
        raw_errors_fixed += 1

    new_record["tiser_normalized_answer"] = normalized_prediction
    new_record["normalized_gold_answer"] = normalized_gold
    new_record["tiser_em"] = normalized_em
    new_record["tiser_f1"] = normalized_f1

    normalized_records.append(new_record)


# Save normalized results
with NORMALIZED_RESULTS_PATH.open("w", encoding="utf-8") as f:
    for record in normalized_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


# Compute metrics
total = len(normalized_records)

overall_em = (
    sum(record["tiser_em"] for record in normalized_records) / total * 100
)

overall_f1 = (
    sum(record["tiser_f1"] for record in normalized_records) / total * 100
)


# Macro metrics by dataset
datasets = {}

for record in normalized_records:
    datasets.setdefault(record["dataset_name"], []).append(record)

dataset_ems = []
dataset_f1s = []

for dataset_records in datasets.values():
    n = len(dataset_records)

    dataset_ems.append(
        sum(record["tiser_em"] for record in dataset_records) / n
    )

    dataset_f1s.append(
        sum(record["tiser_f1"] for record in dataset_records) / n
    )

macro_em = sum(dataset_ems) / len(dataset_ems) * 100
macro_f1 = sum(dataset_f1s) / len(dataset_f1s) * 100


summary = {
    "total_examples": total,
    "tiser_overall_em": overall_em,
    "tiser_overall_token_f1": overall_f1,
    "tiser_macro_em": macro_em,
    "tiser_macro_token_f1": macro_f1,
}


with NORMALIZED_SUMMARY_PATH.open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
    f.write("\n")


print("\n==========================================")
print("LORA + TISER — NORMALIZED FULL TEST SET")
print("==========================================")
print(f"Examples: {total}")
print(f"Predictions changed by normalization: {changed_predictions}")
print(f"Raw EM errors corrected: {raw_errors_fixed}")
print(f"Overall EM: {overall_em:.2f}%")
print(f"Overall F1: {overall_f1:.2f}%")
print(f"Macro EM: {macro_em:.2f}%")
print(f"Macro F1: {macro_f1:.2f}%")
print(f"Results saved to: {NORMALIZED_RESULTS_PATH}")
print(f"Summary saved to: {NORMALIZED_SUMMARY_PATH}")

Loaded examples: 20442

LORA + TISER — NORMALIZED FULL TEST SET
Examples: 20442
Predictions changed by normalization: 166
Raw EM errors corrected: 471
Overall EM: 82.72%
Overall F1: 86.82%
Macro EM: 76.54%
Macro F1: 80.66%
Results saved to: /content/drive/MyDrive/TISER/results/test/lora_both_normalized_results.jsonl
Summary saved to: /content/drive/MyDrive/TISER/results/test/lora_both_normalized_summary.json


In [6]:
import json
import re
from pathlib import Path

from src.evaluate import (
    exact_match,
    token_f1,
    write_combined_summary,
)


INPUT_PATH = Path(
    "/content/drive/MyDrive/TISER/results/test/"
    "lora_both_results.jsonl"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/TISER/results/test"
)

NORMALIZED_RESULTS_PATH = OUTPUT_DIR / "lora_both_normalized_results.jsonl"
NORMALIZED_SUMMARY_PATH = OUTPUT_DIR / "lora_both_normalized_summary.json"


def normalize_answer(prediction, gold_answer):
    prediction = str(prediction).strip()
    gold_answer = str(gold_answer).strip()

    # Duration questions:
    # "1" and "1 year" should be treated equivalently.
    if re.search(r"\s+years?$", gold_answer, flags=re.IGNORECASE):
        prediction = re.sub(
            r"\s+years?$",
            "",
            prediction,
            flags=re.IGNORECASE,
        ).strip()

        gold_answer = re.sub(
            r"\s+years?$",
            "",
            gold_answer,
            flags=re.IGNORECASE,
        ).strip()

    # Event-boundary questions:
    # "(event)" and "(event) starts/ends" should be treated equivalently.
    if re.search(r"\s+(starts|ends)$", gold_answer, flags=re.IGNORECASE):
        prediction = re.sub(
            r"\s+(starts|ends)$",
            "",
            prediction,
            flags=re.IGNORECASE,
        ).strip()

        gold_answer = re.sub(
            r"\s+(starts|ends)$",
            "",
            gold_answer,
            flags=re.IGNORECASE,
        ).strip()

        if prediction.startswith("(") and prediction.endswith(")"):
            prediction = prediction[1:-1].strip()

        if gold_answer.startswith("(") and gold_answer.endswith(")"):
            gold_answer = gold_answer[1:-1].strip()

    return prediction, gold_answer



# Load original combined Direct + TISER results

records = []

with INPUT_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

print(f"Loaded examples: {len(records)}")


# Normalize BOTH Direct and TISER

normalized_records = []

direct_predictions_changed = 0
direct_errors_fixed = 0

tiser_predictions_changed = 0
tiser_errors_fixed = 0


for record in records:
    new_record = dict(record)

    gold_answer = record["gold_answer"]


    # DIRECT

    direct_prediction = record["direct_answer"]

    normalized_direct, direct_gold = normalize_answer(
        direct_prediction,
        gold_answer,
    )

    if normalized_direct != direct_prediction:
        direct_predictions_changed += 1

    normalized_direct_em = exact_match(
        normalized_direct,
        direct_gold,
    )

    normalized_direct_f1 = token_f1(
        normalized_direct,
        direct_gold,
    )

    if not record["direct_em"] and normalized_direct_em:
        direct_errors_fixed += 1



    # TISER

    tiser_prediction = record["tiser_answer"]

    normalized_tiser, tiser_gold = normalize_answer(
        tiser_prediction,
        gold_answer,
    )

    if normalized_tiser != tiser_prediction:
        tiser_predictions_changed += 1

    normalized_tiser_em = exact_match(
        normalized_tiser,
        tiser_gold,
    )

    normalized_tiser_f1 = token_f1(
        normalized_tiser,
        tiser_gold,
    )

    if not record["tiser_em"] and normalized_tiser_em:
        tiser_errors_fixed += 1


    # Save normalized answers separately
    new_record["direct_normalized_answer"] = normalized_direct
    new_record["tiser_normalized_answer"] = normalized_tiser
    new_record["normalized_gold_answer"] = direct_gold

    # Replace Direct metrics with normalized metrics
    new_record["direct_em"] = normalized_direct_em
    new_record["direct_f1"] = normalized_direct_f1

    # Replace TISER metrics with normalized metrics
    new_record["tiser_em"] = normalized_tiser_em
    new_record["tiser_f1"] = normalized_tiser_f1

    normalized_records.append(new_record)



# Save fresh normalized combined JSONL

with NORMALIZED_RESULTS_PATH.open("w", encoding="utf-8") as f:
    for record in normalized_records:
        f.write(
            json.dumps(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )


# Compute fresh summary for BOTH branches


summary = write_combined_summary(
    NORMALIZED_SUMMARY_PATH,
    normalized_records,
)


# Print results


print("\n==============================================")
print("LORA — DIRECT + TISER NORMALIZED FULL TEST")
print("==============================================")
print(f"Examples: {len(normalized_records)}")

print("\nDIRECT")
print(f"Predictions changed: {direct_predictions_changed}")
print(f"Raw EM errors corrected: {direct_errors_fixed}")
print(f"Overall EM: {summary['direct_overall_em']:.2f}%")
print(f"Overall F1: {summary['direct_overall_token_f1']:.2f}%")
print(f"Macro EM: {summary['direct_macro_em']:.2f}%")
print(f"Macro F1: {summary['direct_macro_token_f1']:.2f}%")
print(
    f"Average generated tokens: "
    f"{summary['direct_average_generated_tokens']:.2f}"
)

print("\nTISER")
print(f"Predictions changed: {tiser_predictions_changed}")
print(f"Raw EM errors corrected: {tiser_errors_fixed}")
print(f"Overall EM: {summary['tiser_overall_em']:.2f}%")
print(f"Overall F1: {summary['tiser_overall_token_f1']:.2f}%")
print(f"Macro EM: {summary['tiser_macro_em']:.2f}%")
print(f"Macro F1: {summary['tiser_macro_token_f1']:.2f}%")
print(
    f"Average generated tokens: "
    f"{summary['tiser_average_generated_tokens']:.2f}"
)

print(f"\nResults saved to: {NORMALIZED_RESULTS_PATH}")
print(f"Summary saved to: {NORMALIZED_SUMMARY_PATH}")

Loaded examples: 20442

LORA — DIRECT + TISER NORMALIZED FULL TEST
Examples: 20442

DIRECT
Predictions changed: 67
Raw EM errors corrected: 276
Overall EM: 76.18%
Overall F1: 81.88%
Macro EM: 69.87%
Macro F1: 75.63%
Average generated tokens: 5.96

TISER
Predictions changed: 166
Raw EM errors corrected: 471
Overall EM: 82.72%
Overall F1: 86.82%
Macro EM: 76.54%
Macro F1: 80.66%
Average generated tokens: 283.00

Results saved to: /content/drive/MyDrive/TISER/results/test/lora_both_normalized_results.jsonl
Summary saved to: /content/drive/MyDrive/TISER/results/test/lora_both_normalized_summary.json


In [7]:
import json
from pathlib import Path

RESULTS_PATH = Path(
    "/content/drive/MyDrive/TISER/results/test/"
    "lora_both_normalized_results.jsonl"
)

records = []

with RESULTS_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

total = len(records)

direct_correct_tiser_correct = 0
direct_wrong_tiser_correct = 0
direct_correct_tiser_wrong = 0
direct_wrong_tiser_wrong = 0

for record in records:
    direct_correct = bool(record["direct_em"])
    tiser_correct = bool(record["tiser_em"])

    if direct_correct and tiser_correct:
        direct_correct_tiser_correct += 1

    elif not direct_correct and tiser_correct:
        direct_wrong_tiser_correct += 1

    elif direct_correct and not tiser_correct:
        direct_correct_tiser_wrong += 1

    else:
        direct_wrong_tiser_wrong += 1


def pct(n):
    return 100 * n / total


print(f"Total examples: {total}\n")

print(
    f"Direct correct | TISER correct : "
    f"{direct_correct_tiser_correct} "
    f"({pct(direct_correct_tiser_correct):.2f}%)"
)

print(
    f"Direct wrong   | TISER correct : "
    f"{direct_wrong_tiser_correct} "
    f"({pct(direct_wrong_tiser_correct):.2f}%)"
)

print(
    f"Direct correct | TISER wrong   : "
    f"{direct_correct_tiser_wrong} "
    f"({pct(direct_correct_tiser_wrong):.2f}%)"
)

print(
    f"Direct wrong   | TISER wrong   : "
    f"{direct_wrong_tiser_wrong} "
    f"({pct(direct_wrong_tiser_wrong):.2f}%)"
)

print("\nCheck:", (
    direct_correct_tiser_correct
    + direct_wrong_tiser_correct
    + direct_correct_tiser_wrong
    + direct_wrong_tiser_wrong
))

Total examples: 20442

Direct correct | TISER correct : 14964 (73.20%)
Direct wrong   | TISER correct : 1946 (9.52%)
Direct correct | TISER wrong   : 608 (2.97%)
Direct wrong   | TISER wrong   : 2924 (14.30%)

Check: 20442
